# Pratham AI Worker — Colab T4 (v2)

**Model:** `Qwen/Qwen2.5-Coder-7B-Instruct-AWQ` (AWQ via GPTQModel/ExLlamaV2)  
**Validated environment:** PyTorch 2.11 · Transformers 5.17 · GPTQModel 7.4 · Tesla T4 (14.56 GB VRAM)

### Instructions
1. **Runtime → Change runtime type → T4 GPU** (do this first)
2. Edit **Cell 1** (paste your backend URL + token)
3. **Runtime → Run all**
4. Worker goes ONLINE automatically — you never paste a URL or press reconnect

> The Colab session itself is subject to Google's free-tier limits (~12 h).  
> When a session ends, restart the notebook to bring the worker back.  
> **Pratham AI automatically falls back** to cloud providers while the worker is offline.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — SECRETS  ← ONLY CELL YOU NEED TO EDIT
# ═══════════════════════════════════════════════════════════════════════

# Use the direct production backend URL to avoid 307 redirects across domains:
PRATHAM_BACKEND_URL  = 'https://prathamai.vercel.app'    # direct production URL (no trailing slash)
PRATHAM_WORKER_TOKEN = 'your-new-worker-token-here'    # your rotated secret (matches PRATHAM_WORKER_TOKEN on Vercel)

# ── tunables (safe to leave as-is) ──────────────────────────────────
WORKER_ID                  = 'colab-t4-primary'
MODEL_ID                   = 'Qwen/Qwen2.5-Coder-7B-Instruct-AWQ'  # DO NOT CHANGE
SERVER_PORT                = 7860
HEARTBEAT_INTERVAL_SECONDS = 240   # 4 min; backend timeout window is 6 min
DEFAULT_MAX_NEW_TOKENS     = 2048
DEFAULT_TEMPERATURE        = 0.1   # low for coding accuracy
VERSION                    = '2.0'

print(f'[CONFIG] model={MODEL_ID}  worker_id={WORKER_ID}  port={SERVER_PORT}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — ENVIRONMENT CHECK
# Verifies the runtime matches the validated configuration.
# Aborts with a clear message if requirements are not met.
# ═══════════════════════════════════════════════════════════════════════
import sys, platform

print('─' * 60)
print('Pratham AI Worker — Environment Check')
print('─' * 60)

# Python
py_ver = sys.version_info
print(f'Python       : {platform.python_version()}')
assert py_ver >= (3, 9), f'Python 3.9+ required, got {platform.python_version()}'

# PyTorch + CUDA
import torch
print(f'PyTorch      : {torch.__version__}')
assert torch.cuda.is_available(), 'CUDA not available — make sure Runtime type is set to GPU (T4)'
gpu_name = torch.cuda.get_device_name(0)
vram_total_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print(f'CUDA         : available  ({torch.version.cuda})')
print(f'GPU          : {gpu_name}')
print(f'VRAM         : {vram_total_gb} GB')
assert vram_total_gb >= 12.0, f'Need ≥12 GB VRAM for AWQ model, detected {vram_total_gb} GB'

# Transformers
try:
    import transformers
    print(f'Transformers : {transformers.__version__}')
except ImportError:
    print('Transformers : NOT INSTALLED — will install in Cell 3')

# GPTQModel (provides AWQ/ExLlamaV2 backend)
try:
    import gptqmodel
    print(f'GPTQModel    : {gptqmodel.__version__}')
except ImportError:
    print('GPTQModel    : NOT INSTALLED — will install in Cell 3')

print('─' * 60)
print('Environment check passed ✓')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — INSTALL / UPDATE DEPENDENCIES
# Only installs what is missing or outdated.
# Never installs bitsandbytes — not needed for the AWQ path.
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(args))

# gptqmodel ≥7.0 provides the AWQ + ExLlamaV2 backend
try:
    import gptqmodel
    from packaging.version import Version
    if Version(gptqmodel.__version__) < Version('7.0'):
        raise ImportError('version too old')
    print(f'GPTQModel {gptqmodel.__version__} already installed — skipping')
except ImportError:
    print('Installing gptqmodel...')
    _pip('gptqmodel>=7.0')

# transformers ≥4.40 for Qwen2.5 chat template support
try:
    import transformers
    from packaging.version import Version
    if Version(transformers.__version__) < Version('4.40'):
        raise ImportError('version too old')
    print(f'Transformers {transformers.__version__} already installed — skipping')
except ImportError:
    print('Installing transformers...')
    _pip('transformers>=4.40')

# accelerate (required by transformers device_map)
try:
    import accelerate
    print(f'Accelerate {accelerate.__version__} already installed — skipping')
except ImportError:
    print('Installing accelerate...')
    _pip('accelerate>=0.30')

# flask + requests (inference server + registration)
try:
    import flask, requests
    print(f'Flask {flask.__version__} / Requests {requests.__version__} already installed — skipping')
except ImportError:
    print('Installing flask + requests...')
    _pip('flask>=3.0', 'requests>=2.31')

print('All dependencies ready ✓')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — LOAD MODEL  (runs once; never reloads per request)
# Uses the validated AWQ + GPTQModel/ExLlamaV2 path.
# bitsandbytes is NOT used here.
# ═══════════════════════════════════════════════════════════════════════
import torch
from transformers import AutoTokenizer
from gptqmodel import GPTQModel

_model_ready = False  # flag checked by /ready endpoint

print(f'Loading tokenizer for {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=True,
)
print('Tokenizer loaded ✓')

print(f'Loading {MODEL_ID} via GPTQModel (AWQ / ExLlamaV2 backend)...')
print('  This takes ~60-90 s on first run (model download + GPU init)')
model = GPTQModel.from_quantized(
    MODEL_ID,
    device='cuda:0',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()

# Collect final GPU stats
gpu_info     = torch.cuda.get_device_name(0)
vram_gb      = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
vram_used_gb = round(torch.cuda.memory_allocated(0) / 1e9, 2)
_model_ready = True

print(f'Model loaded ✓  GPU={gpu_info}  VRAM total={vram_gb} GB  used={vram_used_gb} GB')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — INFERENCE SERVER DEFINITION
# Endpoints: GET /health, GET /ready, POST /v1/chat, POST /v1/chat/stream
# GPU concurrency: bounded to 1 via threading.Lock
# Chain-of-thought: NEVER exposed; only safe status events are sent
# ═══════════════════════════════════════════════════════════════════════
import threading, json, time, secrets as _secrets, functools
from flask import Flask, request, Response, stream_with_context
from transformers import TextIteratorStreamer

flask_app   = Flask(__name__)
_gen_lock   = threading.Lock()   # one GPU generation at a time

# ── Auth decorator ───────────────────────────────────────────────────
def _require_token(f):
    @functools.wraps(f)
    def _inner(*a, **kw):
        auth = request.headers.get('Authorization', '')
        tok  = auth[7:].strip() if auth.startswith('Bearer ') else ''
        if not _secrets.compare_digest(tok, PRATHAM_WORKER_TOKEN):
            return Response(
                json.dumps({'type': 'error', 'message': 'Unauthorized'}),
                status=401, content_type='application/json'
            )
        return f(*a, **kw)
    return _inner

# ── SSE helper ───────────────────────────────────────────────────────
def _sse(payload: dict) -> str:
    return f'data: {json.dumps(payload)}\n\n'

# ── /health — always responds, even while model is still loading ──────
@flask_app.route('/health', methods=['GET'])
def route_health():
    return Response(
        json.dumps({
            'status'   : 'ok',
            'worker_id': WORKER_ID,
            'model'    : MODEL_ID,
            'gpu'      : gpu_info,
            'vram_gb'  : vram_gb,
            'ready'    : _model_ready,
        }),
        content_type='application/json'
    )

# ── /ready — returns 200 only after model + tokenizer are fully loaded ─
@flask_app.route('/ready', methods=['GET'])
def route_ready():
    if _model_ready:
        return Response(json.dumps({'ready': True}), content_type='application/json')
    return Response(
        json.dumps({'ready': False, 'message': 'Model still loading'}),
        status=503, content_type='application/json'
    )

# ── Core generation (shared by both routes) ───────────────────────────
def _do_generate(messages, max_new_tokens, temperature, stream=False):
    """
    Returns a generator of SSE strings (stream=True) or
    a (reply_text, error_str) tuple (stream=False).
    The GPU lock is acquired inside this function and released when done.
    """
    _STATUS = [
        'Understanding your request...',
        'Checking the available context...',
        'Generating the response...',
        'Preparing the final answer...',
    ]

    if stream:
        def _gen_stream():
            if not _gen_lock.acquire(blocking=True, timeout=60):
                yield _sse({'type': 'error', 'message': 'GPU busy — please retry in a moment.'})
                return
            try:
                yield _sse({'type': 'thinking', 'message': _STATUS[0]})
                inputs = tokenizer.apply_chat_template(
                    messages,
                    tokenize=True,
                    add_generation_prompt=True,
                    return_dict=True,
                    return_tensors='pt'
                )
                inputs = {k: v.to('cuda:0') for k, v in inputs.items() if torch.is_tensor(v)}
                yield _sse({'type': 'thinking', 'message': _STATUS[2]})
                yield _sse({'type': 'answer_start'})
                streamer = TextIteratorStreamer(
                    tokenizer,
                    skip_prompt=True,
                    skip_special_tokens=True,
                    timeout=30.0,
                )
                gen_kwargs = dict(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=(temperature > 0.01),
                    use_cache=True,
                    streamer=streamer,
                )
                gen_thread = threading.Thread(
                    target=lambda: model.generate(**gen_kwargs),
                    daemon=True
                )
                gen_thread.start()
                for token_text in streamer:
                    if token_text:
                        yield _sse({'type': 'answer_delta', 'text': token_text})
                gen_thread.join(timeout=120)
                yield _sse({'type': 'answer_end'})
            except Exception as exc:
                yield _sse({'type': 'error', 'message': str(exc)})
            finally:
                _gen_lock.release()
        return _gen_stream()

    else:  # non-streaming path
        if not _gen_lock.acquire(blocking=True, timeout=60):
            return None, 'GPU busy — please retry in a moment.'
        try:
            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors='pt'
            )
            inputs = {k: v.to('cuda:0') for k, v in inputs.items() if torch.is_tensor(v)}
            input_length = inputs['input_ids'].shape[1]
            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=(temperature > 0.01),
                    use_cache=True,
                )
            new_ids = outputs[0][input_length:]
            text = tokenizer.decode(new_ids, skip_special_tokens=True)
            return text, None
        except Exception as exc:
            return None, str(exc)
        finally:
            _gen_lock.release()

# ── POST /v1/chat — non-streaming JSON response ───────────────────────
@flask_app.route('/v1/chat', methods=['POST'])
@_require_token
def route_chat():
    if not _model_ready:
        return Response(json.dumps({'error': 'Model not ready yet'}),
                        status=503, content_type='application/json')
    body           = request.get_json(silent=True) or {}
    messages       = body.get('messages', [])
    max_new_tokens = min(int(body.get('max_new_tokens', DEFAULT_MAX_NEW_TOKENS)), 4096)
    temperature    = float(body.get('temperature', DEFAULT_TEMPERATURE))
    text, err = _do_generate(messages, max_new_tokens, temperature, stream=False)
    if err:
        return Response(json.dumps({'error': err}), status=500, content_type='application/json')
    return Response(
        json.dumps({
            'choices': [{'message': {'role': 'assistant', 'content': text}}],
            'text': text,
            'response': text
        }),
        content_type='application/json'
    )

# ── POST /v1/chat/stream — SSE streaming response ─────────────────────
@flask_app.route('/v1/chat/stream', methods=['POST'])
@_require_token
def route_chat_stream():
    if not _model_ready:
        def _not_ready():
            yield _sse({'type': 'error', 'message': 'Model not ready yet — retry in a moment.'})
        return Response(stream_with_context(_not_ready()),
                        content_type='text/event-stream',
                        status=503,
                        headers={'Cache-Control': 'no-cache', 'X-Accel-Buffering': 'no'})
    body           = request.get_json(silent=True) or {}
    messages       = body.get('messages', [])
    max_new_tokens = min(int(body.get('max_new_tokens', DEFAULT_MAX_NEW_TOKENS)), 4096)
    temperature    = float(body.get('temperature', DEFAULT_TEMPERATURE))
    return Response(
        stream_with_context(_do_generate(messages, max_new_tokens, temperature, stream=True)),
        content_type='text/event-stream',
        headers={'Cache-Control': 'no-cache', 'X-Accel-Buffering': 'no'}
    )

def _start_flask():
    import logging
    log = logging.getLogger('werkzeug')
    log.setLevel(logging.ERROR)  # silence request noise in Colab output
    flask_app.run(host='0.0.0.0', port=SERVER_PORT, threaded=True, use_reloader=False)

print('Inference server defined ✓  (Cell 6 will start it)')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — TUNNEL + AUTO-REGISTER + HEARTBEAT LOOP + LIVE DIAGNOSTICS
#
# Full lifecycle (all automatic):
#   Flask starts → tunnel opens → URL detected → live self-tests run
#   → registered with backend → worker ONLINE → heartbeat every 4 min
#
# KEEP THIS CELL RUNNING. Interrupting it takes the worker offline.
# ═══════════════════════════════════════════════════════════════════════
import requests as _http, subprocess, re, time, threading, sys, json

# ── Custom session that preserves Authorization across redirects ─────
class _AuthSession(_http.Session):
    def rebuild_auth(self, prepared_request, response):
        if 'Authorization' not in prepared_request.headers and 'Authorization' in self.headers:
            prepared_request.headers['Authorization'] = self.headers['Authorization']

_session = _AuthSession()

# ── 1. Start Flask in background ─────────────────────────────────────
threading.Thread(target=_start_flask, daemon=True).start()
time.sleep(2)
print(f'[SERVER] Flask listening on port {SERVER_PORT} ✓')

# ── 2. Open tunnel ───────────────────────────────────────────────────
PUBLIC_URL = None

def _try_cloudflare():
    global PUBLIC_URL
    print('[TUNNEL] Trying Cloudflare Quick Tunnel...')
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{SERVER_PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    _url_re = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
    deadline = time.time() + 45
    for line in proc.stdout:
        m = _url_re.search(line)
        if m:
            PUBLIC_URL = m.group(0)
            return proc
        if time.time() > deadline:
            proc.terminate()
            return None
    return None

def _try_ngrok():
    global PUBLIC_URL
    print('[TUNNEL] cloudflared unavailable — installing pyngrok fallback...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
    from pyngrok import ngrok
    tunnel = ngrok.connect(SERVER_PORT)
    PUBLIC_URL = tunnel.public_url
    if PUBLIC_URL.startswith('http://'):
        PUBLIC_URL = PUBLIC_URL.replace('http://', 'https://', 1)

_cf_proc = _try_cloudflare()
if not PUBLIC_URL:
    _try_ngrok()

if not PUBLIC_URL:
    raise RuntimeError('[TUNNEL] Could not open any tunnel. Check cloudflared / pyngrok.')

# ── 3. Configuration & Payloads ──────────────────────────────────────
_BACKEND = PRATHAM_BACKEND_URL.rstrip('/')
_HDRS    = {
    'Authorization': f'Bearer {PRATHAM_WORKER_TOKEN}',
    'Content-Type' : 'application/json',
}
_session.headers.update(_HDRS)

_BASE_PAYLOAD = {
    'worker_id'   : WORKER_ID,
    'endpoint_url': PUBLIC_URL,
    'model'       : MODEL_ID,
    'gpu'         : gpu_info,
    'vram_gb'     : vram_gb,
    'version'     : VERSION,
}

# ── 4. MANDATORY LIVE DIAGNOSTICS & VERIFICATION ──────────────────────
print()
print('═' * 60)
print('COLAB LIVE DIAGNOSTICS')
print(f'PUBLIC_URL          : {PUBLIC_URL}')
print(f'PRATHAM_BACKEND_URL : {_BACKEND}')
print(f'WORKER_ID           : {WORKER_ID}')
print(f'MODEL_ID            : {MODEL_ID}')
print('═' * 60)

# 4A. GET /health
try:
    r_h = _session.get(f'{PUBLIC_URL}/health', timeout=10)
    print(f'GET /health -> HTTP {r_h.status_code}')
except Exception as e:
    print(f'GET /health -> ERROR: {e}')

# 4B. GET /ready
try:
    r_r = _session.get(f'{PUBLIC_URL}/ready', timeout=10)
    print(f'GET /ready -> HTTP {r_r.status_code}')
except Exception as e:
    print(f'GET /ready -> ERROR: {e}')

# 4C. POST /v1/chat — test local generation
test_chat_payload = {
    "messages": [{"role": "user", "content": "Reply with exactly QWEN_WORKER_OK"}],
    "max_new_tokens": 32,
    "temperature": 0.1
}
try:
    r_c = _session.post(f'{PUBLIC_URL}/v1/chat', json=test_chat_payload, timeout=30)
    d_c = r_c.json()
    txt = d_c.get('choices', [{}])[0].get('message', {}).get('content', '') or d_c.get('text', '')
    if 'QWEN_WORKER_OK' in txt:
        print(f'POST /v1/chat -> OK (answer contains QWEN_WORKER_OK) ✓')
    else:
        print(f'POST /v1/chat -> Response text: {txt[:100]}')
except Exception as e:
    print(f'POST /v1/chat -> ERROR: {e}')

# 4D. POST /v1/chat/stream — test real streaming events
try:
    r_s = _session.post(f'{PUBLIC_URL}/v1/chat/stream', json=test_chat_payload, stream=True, timeout=30)
    ev_types = []
    for raw in r_s.iter_lines():
        if not raw: continue
        line = raw.decode('utf-8') if isinstance(raw, bytes) else raw
        if line.startswith('data: '):
            try:
                ev = json.loads(line[6:])
                ev_types.append(ev.get('type'))
            except Exception: pass
    if 'answer_delta' in ev_types and 'answer_end' in ev_types:
        print('STREAM_OK=true ✓')
    else:
        print(f'STREAM events received: {ev_types}')
except Exception as e:
    print(f'POST /v1/chat/stream -> ERROR: {e}')

# 4E. POST /api/worker/register — live register test
try:
    r_reg = _session.post(f'{_BACKEND}/api/worker/register', json=_BASE_PAYLOAD, timeout=15)
    print(f'REGISTER_STATUS={r_reg.status_code}')
    if r_reg.status_code == 200:
        print('REGISTER_OK=true ✓')
    else:
        print(f'REGISTER_FAILED: {r_reg.text[:200]}')
except Exception as e:
    print(f'REGISTER_ERROR: {e}')

# 4F. POST /api/worker/heartbeat — live heartbeat test
try:
    hb_test_payload = dict(_BASE_PAYLOAD, status='online')
    r_hb = _session.post(f'{_BACKEND}/api/worker/heartbeat', json=hb_test_payload, timeout=15)
    print(f'HEARTBEAT_STATUS={r_hb.status_code}')
    if r_hb.status_code == 200:
        print('HEARTBEAT_OK=true ✓')
    else:
        print(f'HEARTBEAT_FAILED: {r_hb.text[:200]}')
except Exception as e:
    print(f'HEARTBEAT_ERROR: {e}')

print('═' * 60)
print(' Worker ONLINE & READY')
print(f' Endpoint       → {PUBLIC_URL}')
print(f' Heartbeat loop → every {HEARTBEAT_INTERVAL_SECONDS}s')
print(' Leave this cell running to keep the worker active.')
print('═' * 60)

# ── 5. Heartbeat loop ────────────────────────────────────────────────
def _heartbeat():
    payload = dict(_BASE_PAYLOAD, status='online')
    try:
        r = _session.post(f'{_BACKEND}/api/worker/heartbeat', json=payload, timeout=12)
        d = r.json()
        print(f'[HB] {"OK" if d.get("ok") else "NOK"} — {d.get("received_at", "?")}')
    except Exception as exc:
        print(f'[HB] error: {exc}')

while True:
    time.sleep(HEARTBEAT_INTERVAL_SECONDS)
    _heartbeat()
